# Demo 02 - Growth Analysis

In this notebook, we want to see how things have changed over time. To do so, we'll connect to the database once more and grab information about buses and invoices.

First, we will install some new libraries to help us work with the data.

<strong>NOTE:</strong> you may need to update the Pandas library you have installed. You can do that by running `pip install --upgrade pandas` in a shell with access to Python.

## Active Buses

The first thing we want to look at is how many active buses the agency has at its disposal in a given year. To simplify the problem, we'll look at the current state as of the beginning of each year.

In [ ]:
import pyodbc
import pandas as pd
import toml

config = toml.load("../config.toml")
server = config["database"]["server"]
database = config["database"]["name"]
username = config["database"]["username"]
password = config["database"]["password"]
conn = pyodbc.connect('DRIVER={ODBC Driver 17 for SQL Server};SERVER='+server+';DATABASE='+database+';UID='+username+';PWD='+password)

In [ ]:
active_buses = pd.read_sql("""SELECT
	c.CalendarYear,
	COUNT(*) AS NumberOfBuses
FROM dbo.Bus b
	INNER JOIN dbo.Calendar c
		ON b.DateFirstInService <= c.Date
		AND ISNULL(b.DateRetired, '2023-12-31') >= c.Date
WHERE
	c.CalendarDayOfYear = 1
	AND c.CalendarYear >= 2011
	AND c.CalendarYear < 2023
GROUP BY
	c.CalendarYear
ORDER BY
	c.CalendarYear;""", conn)

In [ ]:
import seaborn
seaborn.lineplot(data=active_buses, x="CalendarYear", y="NumberOfBuses")

What we can see here is a steady increase over time in the number of buses in our agency's fleet.  As buses rotate out, new ones come in, and this growth is almost constant.

It's reasonable to expect that the number of buses is likely the biggest driver in vehicle expenses, so we'd expect to see a similar growth over time with respect to the number of invoices and expenditures.

## Invoices Per Year

The next thing we want to look at is the number of invoices the agency's staff handle.

In [ ]:
annual_invoices = pd.read_sql("""SELECT
	c.CalendarYear,
	COUNT(*) AS NumberOfInvoices
FROM dbo.LineItem li
	INNER JOIN dbo.Calendar c
		ON li.LineItemDate = c.Date
GROUP BY
	c.CalendarYear
ORDER BY
	c.CalendarYear;""", conn)

In [ ]:
seaborn.lineplot(data=annual_invoices, x="CalendarYear", y="NumberOfInvoices")

This is somewhat strange behavior:  it appears that we have major growth in 2019 and 2020, a flattening in 2021, and a decline in 2022.  Eyeballing these values, it seems a little weird.  I'd now like to compare this to the number of buses on a year-by-year basis.

In [ ]:
# Join together the invoices and buses data frames
invoices_vs_buses = pd.concat([annual_invoices, active_buses], axis=1, join='inner')
# Joining ends up with a duplicate column:  calendar year.  This looks for and removes duplicated columns.
invoices_vs_buses = invoices_vs_buses.loc[:,~invoices_vs_buses.columns.duplicated()]

seaborn.lineplot(data=pd.melt(invoices_vs_buses, "CalendarYear"), x="CalendarYear", y="value", hue="variable")

This shows us a problem with such direct comparisons:  number of invoices and number of buses are on different scales.  To fix that, we can use the `MinMaxScaler()` in the SciKit-Learn package.  This will normalize both sets of data to a range between 0.0 and 1.0.  That way, we can directly compare the two.

In [ ]:
from sklearn import preprocessing
# Normalize invoices and buses
min_max_scaler = preprocessing.MinMaxScaler()
invoices_vs_buses_scaled = pd.DataFrame(min_max_scaler.fit_transform(invoices_vs_buses), columns=invoices_vs_buses.columns, index=invoices_vs_buses.index)
invoices_vs_buses_scaled

As we can see, calendar year is also scaled as part of this, but that's okay--the scale preserves order, so although our calendar year numbers won't be quite right, we will at least get to see some results.

In [ ]:
seaborn.lineplot(data=pd.melt(invoices_vs_buses_scaled, "CalendarYear"), x="CalendarYear", y="value", hue="variable")

What we can see here is that it looks like before 2019, there was a steady increase in buses and a growth in number of invoices which was not quite as strong.  That changed and we have our "hook" shape.  In the final year, the number of invoices returns to something similar to what we'd expect.

Another way to view this data is with what is called a **Quantitle-Quantitle (Q-Q) plot**.  The basic idea of this is, we start with two variables, such as number of buses and number of invoices.  Then, order each variable numerically from smallest to largest.  In our data set, the number of buses is already ordered that way, but the number of invoices gets shuffled around a little.  Finally, we lay out a scatter plot of the ordered sets of points for number of buses and number of invoices.  Optionally, we can draw an identity line--if two things are the same or at least closely related to one another, we will see a fit along this line.

In [ ]:
from seaborn_qqplot import pplot

pplot(data=invoices_vs_buses_scaled, x="NumberOfBuses", y="NumberOfInvoices", kind="qq", aspect=2, height=3, display_kws={"identity":True})

What we see is almost two curves:  one which runs up to about 55-60% of the data and fits **under** the line, and the remainder which fit **over** the line.  This shape isn't necessarily weird, but it is interesting that we see it.

## Expenditures Per Year

Now let's take a brief look at expenditures per year and see if it follows a similar pattern to invoices per year.

In [ ]:
annual_expenditures = pd.read_sql("""SELECT
	c.CalendarYear,
	SUM(li.Amount) AS TotalInvoicedAmount
FROM dbo.LineItem li
	INNER JOIN dbo.Calendar c
		ON li.LineItemDate = c.Date
GROUP BY
	c.CalendarYear
ORDER BY
	c.CalendarYear;""", conn)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

fig, ax = plt.subplots(1, 1, figsize=(6, 4))
annual_expenditures.plot(kind='line', x='CalendarYear', y='TotalInvoicedAmount', ax=ax)
fmt = '${x:,.0f}'
tick = mtick.StrMethodFormatter(fmt)
ax.yaxis.set_major_formatter(tick)
plt.show()

We can see a similar curve in expenditure amounts. This is also a pretty big jump: year-over-year from 2018 into 2019, we saw a jump of approximately 1.5m USD and another 1m USD from 2019 to 2020.

We'll definitely need to do more research to know for sure if this is an issue, but it is certainly interesting.

## Correlation Tests

We've already done one form of correlation testing earlier: the Q-Q test of buses vs expenditures. That test is very powerful but is not the only way to see if two variables are related to one another. We can use a correlation test to do the same work. One nice thing about the correlation test is that we do not need to scale our variables, just mash them together.

### Data by Year

Using the data that we have already, we can correlate variables by year and notice the relationships.

In [ ]:
yearly_data = pd.concat([annual_invoices, active_buses, annual_expenditures], axis=1, join='inner')
# Joining ends up with a duplicate column:  calendar year.  This looks for and removes duplicated columns.
yearly_data = yearly_data.loc[:,~yearly_data.columns.duplicated()]

In [ ]:
yearly_data.corr()

Reading a correlation matrix is fine, but we can also visualize it as a heat map:

In [ ]:
seaborn.heatmap(yearly_data.corr(), annot=True)

Confirming some of our thoughts from earlier:

1. Number of buses and calendar year are almost perfectly correlated.  They are 99.7% correlated, which is pretty close to perfect.
2. Number of invoices and total invoiced amount are almost perfectly correlated.  They are also 99.7% correlated, which is pretty close to perfect.  This says that, at the annual level, a change in one has an almost-exact change in the other.
3. Total invoiced amount is very highly correlated with number of buses, but less strongly than we would have expected.  Also, something which is interesting is that the **number** of invoices is correlated more with number of buses (96.4%) than **invoiced amount** (94.4%).

### Correlating Raw Data

Now let's look at some raw data, as we are more likely to find something interesting at this level.

In [ ]:
raw_data = pd.read_sql("""SELECT
    c.CalendarYear,
    c.CalendarMonth,
    li.BusID,
    b.AverageRoadConditions,
    b.BusModelID,
    bm.BusModel,
    bm.ModelQuality,
    DATEDIFF(YEAR, b.DateFirstInService, li.LineItemDate) AS NumberOfYearsInService,    
    li.ExpenseCategoryID,
    ec.ExpenseCategory,
    li.VendorID,
    v.VendorName,
    li.EmployeeID,
    CONCAT(e.FirstName, ' ', e.LastName) AS EmployeeName,
    li.Amount
FROM dbo.LineItem li
    INNER JOIN dbo.Calendar c
        ON li.LineItemDate = c.Date
    INNER JOIN dbo.Bus b
        ON li.BusID = b.BusID
    INNER JOIN dbo.BusModel bm
        ON b.BusModelID = bm.BusModelID
    INNER JOIN dbo.ExpenseCategory ec
        ON li.ExpenseCategoryID = ec.ExpenseCategoryID
    INNER JOIN dbo.Vendor v
        ON li.VendorID = v.VendorID
    INNER JOIN dbo.Employee e
        ON li.EmployeeID = e.EmployeeID;""", conn)

In [ ]:
raw_data.corr(numeric_only=True)

In [ ]:
plt.figure(figsize=(12,8))
seaborn.heatmap(raw_data.corr(numeric_only=True), annot=True, fmt=".2f")

We can see that most of the relationships have disappeared, although there are a couple which pop out: number of years in service versus bus ID (lower bus IDs have been in service longer), bus model ID vs model quality and vendor ID vs expense category ID. But the latter two are strange relationships because they're looking at **IDs** rather than anything meaningful. Change the surrogate key, change the result.

## Kolmogorov-Smirnov Test

The last analysis we want to look at is called a Kolmogorov-Smirnov Test. It is used in fraud analysis to determine if two sets of data appear to come from the same source.

We noticed some weirdness about the years 2019, 2020, and 2021 versus the other years in the data set. Let's look at this at a bus-by-bus level to get the amount spent on a bus per year, with one set running from 2011-2018 + 2022, and another set running from 2019-2021.

In [ ]:
ks_old = pd.read_sql("""SELECT TOP(2000)
    SUM(li.Amount) AS Amount
FROM dbo.LineItem li
    INNER JOIN dbo.Calendar c
        ON li.LineItemDate = c.[Date]
WHERE
    c.CalendarYear BETWEEN 2011 AND 2018
    OR c.CalendarYear = 2022
GROUP BY
    c.CalendarYear,
    li.BusID
ORDER BY NEWID();""", conn)

ks_new = pd.read_sql("""SELECT TOP(2000)
    SUM(li.Amount) AS Amount
FROM dbo.LineItem li
    INNER JOIN dbo.Calendar c
        ON li.LineItemDate = c.[Date]
WHERE
    c.CalendarYear BETWEEN 2019 AND 2021
GROUP BY
    c.CalendarYear,
    li.BusID
ORDER BY NEWID();""", conn)

There are two versions of the Kolmonov-Smirnov Test: a one-sample version and a two-sample version. The one-sample version is useful when we want to test if one set is greater than the other. For example, if ks_new is definitely greater than ks_old. The two-sample version is if we want to test if ks_new is **different from** ks_old, regardless of whether that means the difference is in one direction or the other.

We will use the two-sample test here.  The output is a statistic and a two-tailed p value.

In [ ]:
from scipy import stats
stats.ks_2samp(ks_old["Amount"], ks_new["Amount"])

Our p value is absolutely tiny. Anything below 0.01 is a decent indicator, but this is quite significant. And the statistic value is 0.305. For a two-sample test, we calculate Dcrit(0.05) as 1.36 * sqrt(1/nx + 1/ny), so it would be `1.36 * sqrt(1/2000 + 1/2000)` or 0.04. If our statistic value is above 0.04, we can expect the distributions to differ, and that statistic is well above the value of 0.04.

This gives us an indication that the distributions are not, in fact, the same. We can see this in action by looking at the empirical cumulative distribution function for each.<br>

In [ ]:
import numpy as np
def ecdf(x):
    xs = np.sort(x)
    ys = np.arange(1, len(xs)+1/float(len(xs)))
    return xs, ys

plt.plot(*ecdf(ks_old["Amount"]))
plt.plot(*ecdf(ks_new["Amount"]))

These curves look quite different, as we see more buses with more expenses in the new data set than the old data set.

Now let's look at 2022 versus 2011-2018. It looked like 2022 brought things back to normal, so let's see what the KS-test says.

In [ ]:
ks_teens = pd.read_sql("""SELECT TOP(1000)
    SUM(li.Amount) AS Amount
FROM dbo.LineItem li
    INNER JOIN dbo.Calendar c
        ON li.LineItemDate = c.[Date]
WHERE
    c.CalendarYear BETWEEN 2011 AND 2018
GROUP BY
    c.CalendarYear,
    li.BusID
ORDER BY NEWID();""", conn)

ks_2022 = pd.read_sql("""SELECT TOP(1000)
    SUM(li.Amount) AS Amount
FROM dbo.LineItem li
    INNER JOIN dbo.Calendar c
        ON li.LineItemDate = c.[Date]
WHERE
    c.CalendarYear = 2022
GROUP BY
    c.CalendarYear,
    li.BusID
ORDER BY NEWID();""", conn)

stats.ks_2samp(ks_teens["Amount"], ks_2022["Amount"])

In [ ]:
plt.plot(*ecdf(ks_teens["Amount"]))
plt.plot(*ecdf(ks_2022["Amount"]))

What this tells us is that there is still a difference from the teens and 2022. The difference is nowhere near as marked as in 2019-2021, but it's still there.

Something I want to make very clear is <strong>THIS IS NOT PROOF OF FRAUD.</strong> The fact that distributions have changed is not, in itself, proof of any sort of malfeasance. There are several reasons outside of fraud which might explain this change over time:

1. Parts might be more expensive now than they were before. If these numbers are not inflation-adjusted, we can see inflation factor into the numbers after several years.  Even if they are inflation-adjusted, market circumstances might change to make certain parts more expensive than before.
2. If older buses are more expensive to care for than newer buses, it might be that the fleet is aging. We know they do not replace their entire fleet each year, so the average age of the bus is perhaps increasing. This is a thing we can test if we want.
3. External conditions may degrade, causing higher expenses. For example, if road conditions deteriorate over time, that might make the average cost per bus higher. If buses need to be driven more than historical norms, that can push the average up. If bus passengers change considerably, that might cause the average cost to increase. For example, suppose riders become considerably more unruly over time; that might cause an increase in repair costs for seating.

At the same time, this <strong>could</strong> be an indicator of fraud. But we're going to need to dig deeper before we can say that for sure.